In [1]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from models.lstm_model import LSTM_Model
from utils import EarlyStopping, get_device, EpochTrainer, build_seq, get_yf_data
from operator import itemgetter
from sklearn.preprocessing import RobustScaler as Scaler
import pandas as pd

# Parameters

In [2]:
model_output_filename = '../checkpoints/lstm_checkpoint.pt'

# Model Config

In [3]:
batch_size = 60
epochs = 100
hidden_size = 64
num_output = 1
num_layers = 2
dropout = 0.2
bidirectional = False
patience = 5
test_size = 0.3
lr = 0.001

eval_method = 'RMSE'
criterion = nn.MSELoss()

In [4]:
steps_ahead = 5
seq_len = 15

start = '2025-01-01'
end   = '2025-12-30'

# Get Data


In [5]:
data_path = f'../financial_data/data/main_{start.replace('-' , '')}_{end.replace('-' , '')}.csv'

data = pd.read_csv(data_path)

In [6]:
input_cols = data.columns.tolist()
input_cols = [col for col in input_cols if col not in ['Close', 'Date']]

identifier_cols = ['Date']
target_cols = ['Close']

# Data Processing

## Scale input and target

In [7]:
x_scaler = Scaler()
y_scaler = Scaler()
    
X_all = x_scaler.fit_transform(data[input_cols].values)
y_all = y_scaler.fit_transform(data[target_cols].values)

## Build Sequence

In [8]:
date_list = data[identifier_cols].values.flatten().tolist()

Seqs = build_seq( seq_len , X_all, steps_ahead, date_list , y_all  )
X_seq, y_seq, X_id, y_id = itemgetter('X_Seq', 'y_Seq', 'X_id', 'y_id')(Seqs)

## Convert to Tensor

In [9]:
X_tensor = torch.from_numpy(np.asarray(X_seq, dtype=np.float32))
y_tensor = torch.tensor(np.array(y_seq), dtype=torch.float32)

In [10]:
full_dataset = TensorDataset(X_tensor, y_tensor)

## Split data into training and testing

In [11]:
input_size = X_tensor.shape[-1]

In [12]:
test_records = int(test_size * len(full_dataset))
train_records = len(full_dataset) - test_records

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_records, test_records],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# Prepare Model

## Prepare model config dictionaries

In [13]:
preprocess_config = {
    "seq_length": seq_len,
    "input_size": input_size,
}

In [14]:
model_config = {
    "input_size": input_size,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_layers": num_layers,
    "dropout": dropout,
    "bidirectional": bidirectional,
}

## Get Model

In [15]:
device = get_device()

In [16]:
model = LSTM_Model(
    input_size=input_size,
    hidden_size=hidden_size,
    num_output=num_output,
    num_layers=num_layers,
    dropout=dropout,
    bidirectional=bidirectional,
).to(device)

In [17]:
optimizer = torch.optim.Adam(model.parameters(), lr = lr)

## Early stopping and checkpoint setup

In [18]:
early_stopping = EarlyStopping(
    patience = patience,
    path = model_output_filename,
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "target_cols": target_cols,
        "input_cols": input_cols,
        "id_cols" : identifier_cols,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "steps_ahead" : steps_ahead,
    },
)

# Loop epochs and train model

In [19]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = eval_method
)

In [20]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():        
        
        print(f"Epoch {epoch + 1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | {key}: {value:.4f}")
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break
        

Epoch   1 | Train Loss: 0.4870 | Val Loss: 0.4680 | RMSE: 0.6841
Epoch   2 | Train Loss: 0.4346 | Val Loss: 0.4407 | RMSE: 0.6639
Epoch   3 | Train Loss: 0.6240 | Val Loss: 0.4130 | RMSE: 0.6427
Epoch   4 | Train Loss: 0.4160 | Val Loss: 0.3782 | RMSE: 0.6150
Epoch   5 | Train Loss: 0.3493 | Val Loss: 0.3388 | RMSE: 0.5820
Epoch   6 | Train Loss: 0.3280 | Val Loss: 0.2932 | RMSE: 0.5415
Epoch   7 | Train Loss: 0.3774 | Val Loss: 0.2416 | RMSE: 0.4915
Epoch   8 | Train Loss: 0.2519 | Val Loss: 0.1987 | RMSE: 0.4458
Epoch   9 | Train Loss: 0.1942 | Val Loss: 0.1903 | RMSE: 0.4362
Epoch  10 | Train Loss: 0.2008 | Val Loss: 0.2099 | RMSE: 0.4582
Epoch  11 | Train Loss: 0.2068 | Val Loss: 0.2111 | RMSE: 0.4595
Epoch  12 | Train Loss: 0.1776 | Val Loss: 0.1892 | RMSE: 0.4350
Epoch  13 | Train Loss: 0.1305 | Val Loss: 0.1669 | RMSE: 0.4085
Epoch  14 | Train Loss: 0.1144 | Val Loss: 0.1481 | RMSE: 0.3848
Epoch  15 | Train Loss: 0.1745 | Val Loss: 0.1333 | RMSE: 0.3651
Epoch  16 | Train Loss: 0